# 🐺 GökTürk-2.5B-Thinking — 6 Aşamalı CoT · Kaggle T4 Eğitimi

**Kaggle ayarları (sağ panel → Settings):**
1. *Accelerator*: **GPU T4 x2** (tek GPU kullanılır)
2. *Internet*: **On** (telefon doğrulaması gerekir)
3. *Add-ons → Secrets*: `HF_TOKEN` (write yetkili) · isteğe bağlı `GEMINI_API_KEY`, `TAVILY_API_KEY` → her birinin yanındaki kutuyu işaretleyip notebook'a bağlayın

**Arka planda çalıştırma:** sağ üst **Save Version → Save & Run All (Commit)**. Hiçbir hücre etkileşim beklemez; sekmeyi kapatabilirsiniz.

Akış: repo → kurulum → secrets → veri → QLoRA eğitim → merge (CPU) → GGUF Q4_K_M + Q8_0 → Hugging Face

In [ ]:
# ⚙️ AYARLAR — yalnızca burayı düzenleyin
MODEL_SIZE   = "3b"        # "3b" (Q4_K_M ≈ 1.9 GB) veya "1.5b" (≈ 1.0 GB, daha hızlı, 2.5 GB RAM'de çok rahat)
EPOCHS       = 2
MAX_STEPS    = -1          # hızlı deneme için 60 yapın
QUANTS       = "q4_k_m q8_0"
PUSH_TO_HF   = True        # HF_TOKEN secret'ı gerekir
HF_REPO      = ""          # boş → <hf_kullanıcı_adınız>/GokTurk-2.5B-Thinking
USE_DISTILL  = False       # True: GEMINI_API_KEY (+TAVILY_API_KEY) ile ek yüksek kaliteli veri üret
DISTILL_TOPICS = 300       # distill için konu havuzundan üretilecek soru sayısı
BRANCH       = "arena/01a0d4d3-gokturk-2-5b-thinking"
REPO_URL     = "https://github.com/a23521745-hub/GokTurk-2.5B-Thinking.git"
REPO_DIR     = "/kaggle/working/GokTurk-2.5B-Thinking"

In [ ]:
# 📥 Repo
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "-q"])
os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "log", "--oneline", "-1"]).decode())

In [ ]:
# 📦 Kurulum (unsloth; torch/triton/xformers/trl/peft uyumlu sürümlerle gelir) — 3-5 dk
!pip install -q unsloth
!pip install -q --upgrade huggingface_hub openai datasets
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 🔑 Kaggle Secrets → ortam değişkenleri (alt süreçlere aktarılır; değerler yazdırılmaz)
import sys; sys.path.insert(0, REPO_DIR)
import gokturk_env
status = gokturk_env.load_secrets()
if PUSH_TO_HF and not status["HF_TOKEN"]:
    print("⚠️ HF_TOKEN bağlı değil → eğitim yapılacak ama yükleme atlanacak.")

In [ ]:
# 📚 Veri: (1) /kaggle/input'a eklenmiş train.jsonl  (2) sentetik  (3) isteğe bağlı Gemini+Tavily distill
import glob, shutil
os.makedirs("data/gokturk6", exist_ok=True)
attached = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
if attached:
    shutil.copy(attached[0], "data/gokturk6/train.jsonl")
    val = attached[0].replace("train.jsonl", "val.jsonl")
    if os.path.exists(val): shutil.copy(val, "data/gokturk6/val.jsonl")
    print("✓ Ekli veri kullanılıyor:", attached[0])
else:
    parts = ["data/synth6.jsonl"]
    !python prepare_dataset.py synthetic -o data/synth6.jsonl
    extra = sorted(f for f in glob.glob("/kaggle/input/**/*.jsonl", recursive=True) if "seeds" not in f)
    parts += extra
    if USE_DISTILL and status["GEMINI_API_KEY"]:
        !python prepare_dataset.py distill --topics {DISTILL_TOPICS} --workers 6 -o data/distill6.jsonl
        parts.append("data/distill6.jsonl")
    !python prepare_dataset.py merge {" ".join(parts)} -o data/gokturk6

In [ ]:
# 🚀 Eğitim (3B, ~2000 örnek, 2 epoch ≈ 45-70 dk)
!python train_unsloth.py --size {MODEL_SIZE} --epochs {EPOCHS} --max-steps {MAX_STEPS}

In [ ]:
# 🔗 Merge (CPU) → GGUF → Hugging Face
push_flags = "--push" + (f" --repo {HF_REPO}" if HF_REPO else "") if (PUSH_TO_HF and status["HF_TOKEN"]) else ""
!python push_to_hf.py --quants {QUANTS} {push_flags}

In [ ]:
# 📦 Çıktılar (Kaggle → Output sekmesinden de indirilebilir)
for root, _, files in os.walk("/kaggle/working/outputs"):
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p)/1024**2:10.1f} MB  {p}")